In [1]:
"""
Step 5: Model Development
==========================
"""

import numpy as np
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"\n✓ TensorFlow version: {tf.__version__}")
print(f"✓ GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")


✓ TensorFlow version: 2.20.0
✓ GPU Available: False


In [2]:
# 1. LOAD METADATA AND DATA

print("\n1. LOADING DATA AND METADATA")
print("-" * 80)

with open('../data/processed/metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

n_features = metadata['n_features']
n_targets = metadata['n_targets']
max_seq_length = metadata['max_sequence_length']

print(f"✓ Input features: {n_features}")
print(f"✓ Output targets: {n_targets}")
print(f"✓ Max sequence length: {max_seq_length}")

train_data = np.load(
    '../data/processed/train_data.npz',
    allow_pickle=True
)

X_train = train_data['X']
y_train = train_data['y']

print(f"✓ Training data shape: {X_train.shape}")
print(f"✓ Target data shape: {y_train.shape}")



1. LOADING DATA AND METADATA
--------------------------------------------------------------------------------
✓ Input features: 12
✓ Output targets: 2
✓ Max sequence length: 10
✓ Training data shape: (177322, 10, 12)
✓ Target data shape: (177322, 2)


In [3]:
# 2. MODEL ARCHITECTURE 1: LSTM-BASED MODEL

print("\n2. BUILDING MODEL 1: LSTM-BASED ARCHITECTURE")
print("-" * 80)

def build_lstm_model(input_shape, n_targets):
    """
    LSTM-based model for sequential growth prediction
    
    Architecture:
    - LSTM layers to capture temporal patterns
    - Dropout for regularization
    - Dense layers for final prediction
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.LSTM(128, return_sequences=True, name='lstm_1'),
        layers.Dropout(0.2),
        
        layers.LSTM(64, return_sequences=False, name='lstm_2'),
        layers.Dropout(0.2),
        
        layers.Dense(64, activation='relu', name='dense_1'),
        layers.Dropout(0.1),
        
        layers.Dense(32, activation='relu', name='dense_2'),
        
        layers.Dense(n_targets, activation='linear', name='output')
    ])
    
    return model

lstm_model = build_lstm_model((max_seq_length, n_features), n_targets)

print("✓ LSTM Model Architecture:")
lstm_model.summary()


2. BUILDING MODEL 1: LSTM-BASED ARCHITECTURE
--------------------------------------------------------------------------------
✓ LSTM Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 10, 128)        │        72,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 127,906 (499.63 KB)

 Trainable params: 127,906 (499.63 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# 3. MODEL ARCHITECTURE 2: GRU-BASED MODEL

print("\n3. BUILDING MODEL 2: GRU-BASED ARCHITECTURE")
print("-" * 80)

def build_gru_model(input_shape, n_targets):
    """
    GRU-based model (lighter alternative to LSTM)
    
    Architecture:
    - GRU layers for temporal modeling
    - Faster training than LSTM
    - Similar performance
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.GRU(128, return_sequences=True, name='gru_1'),
        layers.Dropout(0.2),
        
        layers.GRU(64, return_sequences=False, name='gru_2'),
        layers.Dropout(0.2),
        
        layers.Dense(64, activation='relu', name='dense_1'),
        layers.Dropout(0.1),
        
        layers.Dense(32, activation='relu', name='dense_2'),
        
        layers.Dense(n_targets, activation='linear', name='output')
    ])
    
    return model

gru_model = build_gru_model((max_seq_length, n_features), n_targets)

print("✓ GRU Model Architecture:")
gru_model.summary()



3. BUILDING MODEL 2: GRU-BASED ARCHITECTURE
--------------------------------------------------------------------------------
✓ GRU Model Architecture:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 10, 128)        │        54,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,082 (383.13 KB)

 Trainable params: 98,082 (383.13 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# 4. MODEL ARCHITECTURE 3: BIDIRECTIONAL LSTM

print("\n4. BUILDING MODEL 3: BIDIRECTIONAL LSTM")
print("-" * 80)

def build_bidirectional_lstm_model(input_shape, n_targets):
    """
    Bidirectional LSTM model
    
    Architecture:
    - Bidirectional processing for better context understanding
    - Higher capacity but more computational cost
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.Bidirectional(
            layers.LSTM(64, return_sequences=True),
            name='bi_lstm_1'
        ),
        layers.Dropout(0.2),
        
        layers.Bidirectional(
            layers.LSTM(32, return_sequences=False),
            name='bi_lstm_2'
        ),
        layers.Dropout(0.2),
        
        layers.Dense(64, activation='relu', name='dense_1'),
        layers.Dropout(0.1),
        
        layers.Dense(32, activation='relu', name='dense_2'),
        
        layers.Dense(n_targets, activation='linear', name='output')
    ])
    
    return model

bi_lstm_model = build_bidirectional_lstm_model((max_seq_length, n_features), n_targets)

print("✓ Bidirectional LSTM Model Architecture:")
bi_lstm_model.summary()


4. BUILDING MODEL 3: BIDIRECTIONAL LSTM
--------------------------------------------------------------------------------
✓ Bidirectional LSTM Model Architecture:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bi_lstm_1 (Bidirectional)       │ (None, 10, 128)        │        39,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_lstm_2 (Bidirectional)       │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 86,946 (339.63 KB)

 Trainable params: 86,946 (339.63 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# 5. MODEL ARCHITECTURE 4: CNN-LSTM HYBRID

print("\n5. BUILDING MODEL 4: CNN-LSTM HYBRID")
print("-" * 80)

def build_cnn_lstm_model(input_shape, n_targets):
    """
    Hybrid CNN-LSTM model
    
    Architecture:
    - 1D CNN for feature extraction
    - LSTM for temporal modeling
    - Combines strengths of both
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same', name='conv_1'),
        layers.MaxPooling1D(pool_size=2, name='pool_1'),
        layers.Dropout(0.2),
        
        layers.Conv1D(32, kernel_size=3, activation='relu', padding='same', name='conv_2'),
        
        layers.LSTM(64, return_sequences=False, name='lstm_1'),
        layers.Dropout(0.2),
        
        layers.Dense(64, activation='relu', name='dense_1'),
        layers.Dropout(0.1),
        
        layers.Dense(32, activation='relu', name='dense_2'),
        
        layers.Dense(n_targets, activation='linear', name='output')
    ])
    
    return model

cnn_lstm_model = build_cnn_lstm_model((max_seq_length, n_features), n_targets)

print("✓ CNN-LSTM Hybrid Model Architecture:")
cnn_lstm_model.summary()


5. BUILDING MODEL 4: CNN-LSTM HYBRID
--------------------------------------------------------------------------------
✓ CNN-LSTM Hybrid Model Architecture:


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_1 (Conv1D)                 │ (None, 10, 64)         │         2,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv1D)                 │ (None, 5, 32)          │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,682 (155.01 KB)

 Trainable params: 39,682 (155.01 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# 6. MODEL ARCHITECTURE 5: ATTENTION-BASED LSTM

print("\n6. BUILDING MODEL 5: ATTENTION-BASED LSTM")
print("-" * 80)

class AttentionLayer(layers.Layer):
    """Custom attention layer"""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    
    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)
    
    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

def build_attention_lstm_model(input_shape, n_targets):
    """
    LSTM with attention mechanism
    
    Architecture:
    - LSTM layers with return_sequences
    - Attention layer to focus on important timesteps
    - Dense layers for prediction
    """
    inputs = layers.Input(shape=input_shape)
    
    x = layers.LSTM(128, return_sequences=True, name='lstm_1')(inputs)
    x = layers.Dropout(0.2)(x)
    
    x = layers.LSTM(64, return_sequences=True, name='lstm_2')(x)
    x = layers.Dropout(0.2)(x)
    
    x = AttentionLayer(name='attention')(x)
    
    x = layers.Dense(64, activation='relu', name='dense_1')(x)
    x = layers.Dropout(0.1)(x)
    
    x = layers.Dense(32, activation='relu', name='dense_2')(x)
    
    outputs = layers.Dense(n_targets, activation='linear', name='output')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

attention_model = build_attention_lstm_model((max_seq_length, n_features), n_targets)

print("✓ Attention-based LSTM Model Architecture:")
attention_model.summary()


6. BUILDING MODEL 5: ATTENTION-BASED LSTM
--------------------------------------------------------------------------------
✓ Attention-based LSTM Model Architecture:


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 10, 12)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10, 128)        │        72,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 10, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (AttentionLayer)      │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 132,066 (515.88 KB)

 Trainable params: 132,066 (515.88 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# 7. COMPILE ALL MODELS

print("\n7. COMPILING MODELS")
print("-" * 80)

def compile_model(model, learning_rate=0.001):
    """Compile model with appropriate loss and metrics"""
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss='mse',
        metrics=[
            'mae',
            tf.keras.metrics.RootMeanSquaredError(name='rmse')
        ]
    )
    return model

models_dict = {
    'LSTM': compile_model(lstm_model),
    'GRU': compile_model(gru_model),
    'Bidirectional_LSTM': compile_model(bi_lstm_model),
    'CNN_LSTM': compile_model(cnn_lstm_model),
    'Attention_LSTM': compile_model(attention_model)
}

print("✓ All models compiled with:")
print("  - Optimizer: Adam (lr=0.001)")
print("  - Loss: MSE (Mean Squared Error)")
print("  - Metrics: MAE, RMSE")


7. COMPILING MODELS
--------------------------------------------------------------------------------
✓ All models compiled with:
  - Optimizer: Adam (lr=0.001)
  - Loss: MSE (Mean Squared Error)
  - Metrics: MAE, RMSE


In [9]:
# 8. SAVE MODEL ARCHITECTURES

print("\n8. SAVING MODEL ARCHITECTURES")
print("-" * 80)

for name, model in models_dict.items():
    model_path = f'../data/models/{name}_architecture.keras'
    model.save(model_path)
    print(f"✓ Saved {name} to: {model_path}")

model_info = {
    'models': list(models_dict.keys()),
    'input_shape': (max_seq_length, n_features),
    'output_shape': n_targets,
    'target_names': ['height', 'weight']
}

with open('../data/models/model_info.pkl', 'wb') as f:
    pickle.dump(model_info, f)

print("✓ Saved model information to: ..data/models/model_info.pkl")


8. SAVING MODEL ARCHITECTURES
--------------------------------------------------------------------------------
✓ Saved LSTM to: ../data/models/LSTM_architecture.keras
✓ Saved GRU to: ../data/models/GRU_architecture.keras
✓ Saved Bidirectional_LSTM to: ../data/models/Bidirectional_LSTM_architecture.keras
✓ Saved CNN_LSTM to: ../data/models/CNN_LSTM_architecture.keras
✓ Saved Attention_LSTM to: ../data/models/Attention_LSTM_architecture.keras
✓ Saved model information to: ..data/models/model_info.pkl


In [10]:
# 9. MODEL COMPARISON TABLE

print("\n9. MODEL COMPARISON")
print("-" * 80)

print("\n{:<25} {:<15} {:<15}".format("Model", "Parameters", "Type"))
print("-" * 55)

for name, model in models_dict.items():
    params = model.count_params()
    model_type = "Sequential" if isinstance(model, models.Sequential) else "Functional"
    print("{:<25} {:<15,} {:<15}".format(name, params, model_type))


9. MODEL COMPARISON
--------------------------------------------------------------------------------

Model                     Parameters      Type           
-------------------------------------------------------
LSTM                      127,906         Sequential     
GRU                       98,082          Sequential     
Bidirectional_LSTM        86,946          Sequential     
CNN_LSTM                  39,682          Sequential     
Attention_LSTM            132,066         Functional     


In [11]:
# 10. MODEL DEVELOPMENT SUMMARY

print("\n" + "=" * 80)
print("MODEL DEVELOPMENT SUMMARY")
print("=" * 80)

print("\n✓ 5 Model Architectures Created:")
print("  1. LSTM - Standard sequential model")
print("  2. GRU - Lighter alternative to LSTM")
print("  3. Bidirectional LSTM - Better context understanding")
print("  4. CNN-LSTM Hybrid - Feature extraction + temporal modeling")
print("  5. Attention-LSTM - Focus on important timesteps")

print(f"\n✓ Input Shape: ({max_seq_length}, {n_features})")
print(f"✓ Output Shape: {n_targets} (height, weight)")

print("\n✓ All models ready for training")
print("✓ Model files saved to: data/models/")


MODEL DEVELOPMENT SUMMARY

✓ 5 Model Architectures Created:
  1. LSTM - Standard sequential model
  2. GRU - Lighter alternative to LSTM
  3. Bidirectional LSTM - Better context understanding
  4. CNN-LSTM Hybrid - Feature extraction + temporal modeling
  5. Attention-LSTM - Focus on important timesteps

✓ Input Shape: (10, 12)
✓ Output Shape: 2 (height, weight)

✓ All models ready for training
✓ Model files saved to: data/models/
